In [1]:
from discovery_utils.utils import search
from discovery_utils.getters import gtr

from discovery_utils import PROJECT_DIR
VECTOR_DB_DIR = PROJECT_DIR / 'tmp/vector_db'

GTR = gtr.GtrGetter(vector_db_path=VECTOR_DB_DIR)

/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-11-19 14:46:37,376 - discovery_utils.getters.gtr - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2024-11-19 14:46:37,451 - discovery_utils.getters.gtr - INFO - Latest version found: GtR_20241117


In [2]:
Search = search.SearchDataset(GTR, GTR.projects_enriched, "config.yaml")

2024-11-19 14:46:38,014 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20241117/projects.parquet
2024-11-19 14:47:06,315 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20241117/projects.parquet
2024-11-19 14:47:07,581 - discovery_utils.getters.gtr - INFO - Downloading parquet file: data/GtR/GtR_20241117/funds.parquet
2024-11-19 14:47:10,487 - discovery_utils.getters.gtr - INFO - Successfully downloaded and read parquet file: data/GtR/GtR_20241117/funds.parquet


In [3]:
search_df = Search.do_search()

2024-11-19 14:47:18,751 - root - INFO - Folder /Users/karlis.kanders/Code/discovery_utils/tmp/vector_db/gtr-lancedb/ already exists. Set overwrite=True to download again.
2024-11-19 14:47:18,754 - root - INFO - Connected with database gtr-lancedb. Available tables: ['project_embeddings']
2024-11-19 14:47:18,765 - root - ERROR - Error creating FTS index: Index already exists. Use replace=True to overwrite.
2024-11-19 14:47:20,966 - discovery_utils - INFO - Keyword search found 80 matches
2024-11-19 14:47:20,967 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2
/Users/karlis.kanders/Code/discovery_utils/.venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/hug

In [87]:
from discovery_utils.utils import (
    analysis,
    analysis_gtr,
)
import importlib
importlib.reload(analysis);
importlib.reload(analysis_gtr);

In [66]:
df = (
    search_df
    .merge(GTR.get_projects_text(), on='id', how='left')
    .query("_score_avg > 0.3")
)
print(len(df))
df_dedup = analysis_gtr.deduplicate_projects(df, description_column='text')
print(len(df_dedup))

95
91


In [112]:
analysis_gtr.funding_per_period(df, period='year', min_year=2010, max_year=2024)

,time_period,year,n_projects,amount,amount_median
0,2010-01-01,2010,0,0.0,0.0
1,2011-01-01,2011,1,47602.0,47602.0
2,2012-01-01,2012,1,100347.0,100347.0
3,2013-01-01,2013,2,195167.0,97583.5
4,2014-01-01,2014,5,432421.0,64130.0
5,2015-01-01,2015,6,293760.0,5000.0
6,2016-01-01,2016,6,1452313.0,5000.0
7,2017-01-01,2017,3,1605978.0,659952.0
8,2018-01-01,2018,11,1745988.0,0.0
9,2019-01-01,2019,5,222819.0,14950.0


In [113]:
ts_df = analysis_gtr.get_timeseries(df, period='year', min_year=2010, max_year=2024)
analysis.magnitude_growth(ts_df, 2019, 2024)
# ts_df

,magnitude,growth
n_projects,9.333333e+00,35.294118
amount,2.621404e+06,226.458738
amount_median,1.611431e+05,-3.458724


In [114]:
from discovery_utils.getters import crunchbase
CB = crunchbase.CrunchbaseGetter(vector_db_path=VECTOR_DB_DIR)

2024-11-19 15:50:46,425 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2024-11-19 15:50:46,641 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2024-11-18
2024-11-19 15:50:46,641 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/organizations_full.parquet
2024-11-19 15:52:57,134 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/enriched/organizations_full.parquet
2024-11-19 15:52:57,166 - root - INFO - Folder /Users/karlis.kanders/Code/discovery_utils/tmp/vector_db/crunchbase-lancedb/ already exists. Set overwrite=True to download again.
2024-11-19 15:52:57,175 - root - INFO - Connected with database crunchbase-lancedb. Available tables: ['company_embeddings']
2024-11-19 15:52:57,180 - root - ERROR - Error creating FTS index: Index already exists. Use replace=True to overwrite.
2

In [ ]:
SearchCB = search.SearchDataset(CB, CB.organisations_enriched, "config.yaml")
search_cb_df = SearchCB.do_search()

In [129]:
orgs_df = CB.organisations_enriched.copy()
funds_df = CB.funding_rounds_enriched.copy()

2024-11-19 16:20:22,349 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/funding_rounds_full.parquet
2024-11-19 16:20:44,738 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/enriched/funding_rounds_full.parquet


In [116]:
df = (
    search_cb_df
    .query("_score_avg > 0.3")
)

In [204]:
from discovery_utils.utils import (
    analysis,
    analysis_crunchbase,
)
importlib.reload(analysis);
importlib.reload(analysis_crunchbase);

In [198]:
analysis_crunchbase.orgs_founded_per_period(df, 'year', 2010, 2024)

,time_period,n_orgs_founded
0,2010-01-01,5
1,2011-01-01,2
2,2012-01-01,4
3,2013-01-01,10
4,2014-01-01,14
5,2015-01-01,5
6,2016-01-01,6
7,2017-01-01,7
8,2018-01-01,6
9,2019-01-01,7


In [133]:
matching_ids = df.id.to_list()

In [134]:
selected_funding_df = (
    CB.funding_rounds_enriched
    .query("org_id in @matching_ids")
    .query(f"year >= {2010}")
    .query(f"year <= {2024}")
    # .query(f"investment_type in @include_deals")
    .drop_duplicates("funding_round_id")
)

In [191]:
importlib.reload(analysis_crunchbase);

In [ ]:
analysis_crunchbase

In [205]:
deals_df, deal_counts_df = analysis_crunchbase.get_funding_by_year_and_range(selected_funding_df, 2014, 2024)
deals_df

,year,n/a,£0-5M,£5-20M,£20-100M,£100M+,total_amount
0,2014,0.0,1.702734,0.000000,0.000000,0.0,1.702734
1,2015,0.0,2.020709,0.000000,0.000000,0.0,2.020709
2,2016,0.0,2.742130,0.000000,0.000000,0.0,2.742130
3,2017,0.0,0.565402,0.000000,0.000000,0.0,0.565402
4,2018,0.0,0.430524,9.768104,0.000000,0.0,10.198627
5,2019,0.0,0.000000,0.000000,0.000000,0.0,0.000000
6,2020,0.0,0.092103,0.000000,0.000000,0.0,0.092103
7,2021,0.0,1.580690,0.000000,72.398815,0.0,73.979504
8,2022,0.0,2.712131,10.300441,0.000000,0.0,13.012572
9,2023,0.0,2.497819,0.000000,0.000000,0.0,2.497819


In [206]:
deal_counts_df

,year,n/a,£0-5M,£5-20M,£20-100M,£100M+,total_counts
0,2014,3,4,0,0,0,7
1,2015,3,3,0,0,0,6
2,2016,1,4,0,0,0,5
3,2017,0,3,0,0,0,3
4,2018,0,2,1,0,0,3
5,2019,2,0,0,0,0,2
6,2020,3,1,0,0,0,4
7,2021,2,3,0,1,0,6
8,2022,3,2,1,0,0,6
9,2023,3,4,0,0,0,7


In [207]:
(
    selected_funding_df
    .query("announced_on >= '2015-01-01'")
    .query("announced_on < '2016-01-01'")
    .raised_amount_gbp.sum()
)

2020.709340291793

In [212]:
importlib.reload(analysis_crunchbase);
ts_df = analysis_crunchbase.get_timeseries(df, selected_funding_df, 'year', 2010, 2024)

In [213]:
ts_df

,time_period,year,n_rounds,raised_amount_usd_total,raised_amount_gbp_total,n_orgs_founded
0,2010-01-01,2010,5,3.073000,1.929978,5
1,2011-01-01,2011,6,5.775000,3.608702,2
2,2012-01-01,2012,5,1.467900,0.925512,4
3,2013-01-01,2013,2,0.015000,0.009968,10
4,2014-01-01,2014,7,2.890469,1.702734,14
5,2015-01-01,2015,6,3.067977,2.020709,5
6,2016-01-01,2016,5,3.912430,2.742130,6
7,2017-01-01,2017,3,0.734822,0.565402,7
8,2018-01-01,2018,3,13.547525,10.198627,6
9,2019-01-01,2019,2,0.000000,0.000000,7


In [215]:
analysis.magnitude_growth(ts_df, 2019, 2024)

,magnitude,growth
n_rounds,4.500000,87.500000
raised_amount_usd_total,20.463454,43.828315
raised_amount_gbp_total,14.930333,44.094649
n_orgs_founded,4.166667,-90.000000
